# Video 09 - Exercise Solutions

Try `../exercises.md` first. Spoilers below.

Run the Connect + helper cells from the learner notebook (or the setup cell) first.


In [ ]:
import json, time
from datetime import date
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
ORDERS = {42: {"customer": "Alice", "total": 250.0}, 7: {"customer": "Bob", "total": 80.0}}

def get_order_total(customer_id: int):
    order = ORDERS.get(int(customer_id))
    return order["total"] if order else "customer not found"

def calculator(expression: str):
    if not set(expression) <= set("0123456789+-*/(). "):
        return "error: only basic arithmetic is allowed"
    return eval(expression, {"__builtins__": {}})

def get_today():
    return date.today().isoformat()

def tool_spec(name, description, properties=None, required=None):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties or {}, "required": required or []}}}

TOOLS = {"get_order_total": get_order_total, "calculator": calculator, "get_today": get_today}
WRITE_TOOLS = {"cancel_order"}
TOOL_SPECS = [
    tool_spec("get_order_total", "Look up the order total in dollars for a customer id.",
              {"customer_id": {"type": "integer"}}, ["customer_id"]),
    tool_spec("calculator", "Evaluate a basic arithmetic expression.",
              {"expression": {"type": "string"}}, ["expression"]),
    tool_spec("get_today", "Get today's date in ISO format."),
]

def run_tool_gated(name, args, functions, allow_writes=False):
    if name not in functions:
        return f"error: unknown tool {name}"
    if name in WRITE_TOOLS and not allow_writes:
        return "error: write requires approval"
    if name == "calculator":
        expr = str(args.get("expression", ""))
        if len(expr) > 40 or any(c.isalpha() for c in expr):
            return "error: invalid calculator expression"
    try:
        result = functions[name](**args)
        return result if isinstance(result, str) else json.dumps(result)
    except Exception as e:
        return f"error: {e}"

print("Setup ready.")


In [ ]:
def run_safe_agent_timed(question, max_steps=5, max_seconds=30.0, **kwargs):
    start = time.time()
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        if time.time() - start > max_seconds:
            return f"Stopped: time budget exceeded ({max_seconds}s)."
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            content = run_tool_gated(call.function.name, args, TOOLS)
            print(f"  step {step}: {call.function.name}({args}) -> {content}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})
    return "Stopped: max_steps."

print(run_safe_agent_timed("What's 15% of the order total for customer 42?", max_seconds=0.01))


In [ ]:
print(run_tool_gated("calculator", {"expression": 'os.system("ls")'}, TOOLS))
print(run_tool_gated("calculator", {"expression": "250 * 0.15"}, TOOLS))


In [ ]:
def ask_with_truncation_retry(messages, max_tokens=16):
    response = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens)
    if response.choices[0].finish_reason != "length":
        return response.choices[0].message.content, max_tokens, False
    # one soft retry
    response2 = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens * 4)
    truncated = response2.choices[0].finish_reason == "length"
    return response2.choices[0].message.content, max_tokens * 4, truncated

msgs = [
    {"role": "system", "content": "Answer in one short sentence."},
    {"role": "user", "content": "What is an agent loop?"},
]
text, used, still_cut = ask_with_truncation_retry(msgs, max_tokens=8)
print("max_tokens used:", used, "still truncated:", still_cut)
print("text:", text)


In [ ]:
def cancel_order(order_id: int):
    return f"cancelled order {order_id}"

ALL = {**TOOLS, "cancel_order": cancel_order}
print(run_tool_gated("cancel_order", {"order_id": 42}, ALL, allow_writes=False))
print(run_tool_gated("cancel_order", {"order_id": 42}, ALL, allow_writes=True))
